# BioDCASE 2026 — Cross-Domain Mosquito Species Classification (CD-MSC)
## Comprehensive Exploratory Data Analysis

**Dataset:** 271,380 audio clips | 60.66 hours | 9 species | 5 domains  
**Source:** [Zenodo Record 19095788](https://zenodo.org/records/19095788)  
**Baseline:** MTRCNN (0.22M params) — BA_seen=0.8806, BA_unseen=0.1751, DSG=0.7055  

---

### EDA Workflow
1. **Data Inspection & Understanding** — Load, parse filenames, check shape/types/summary stats
2. **Univariate Analysis** — Distribution of species, domains, durations, frequencies
3. **Bivariate & Multivariate Analysis** — Species×Domain interactions, spectral comparisons
4. **Cleaning & Preprocessing Insights** — Imbalance quantification, domain shift evidence

---
## 0. Setup & Dependencies

In [ ]:
!pip install -q librosa matplotlib seaborn pandas numpy scipy scikit-learn tqdm ipywidgets

In [ ]:
import os, re, warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('All imports successful ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════
# DATA ROOT — GCP Instance Path
# The zip extracts with a nested folder structure:
#   ~/Development_data/Development_data/*.wav
# ══════════════════════════════════════════════════════════════

DATA_ROOT = Path.home() / 'Development_data' / 'Development_data'

# Fallback: try common alternative paths
if not DATA_ROOT.exists():
    alternatives = [
        Path.home() / 'Development_data',
        Path('/home/jupyter/Development_data/Development_data'),
        Path('/home/jupyter/Development_data'),
        Path('.') / 'Development_data' / 'Development_data',
        Path('.') / 'Development_data',
    ]
    for alt in alternatives:
        if alt.exists() and list(alt.glob('*.wav')):
            DATA_ROOT = alt
            break

assert DATA_ROOT.exists(), f'Data root not found: {DATA_ROOT}\nPlease update DATA_ROOT'

wav_files = sorted(DATA_ROOT.glob('*.wav'))
print(f'DATA_ROOT : {DATA_ROOT}')
print(f'WAV files : {len(wav_files):,}')
print(f'Sample    : {wav_files[0].name if wav_files else "NONE"}')

---
## 1. Data Inspection & Understanding
Parse the filename convention `S_<speciesID>_D_<domainID>_<clipIndex>.wav`

In [ ]:
# ── Species ID → Scientific Name Mapping ──
SPECIES_MAP = {
    1: 'Ae. aegypti',
    2: 'Ae. albopictus',
    3: 'Cx. quinquefasciatus',
    4: 'An. gambiae',
    5: 'An. arabiensis',
    6: 'An. dirus',
    7: 'Cx. pipiens',
    8: 'An. minimus',
    9: 'An. stephensi',
}

DOMAIN_MAP = {i: f'D{i}' for i in range(1, 6)}

# ── Parse every filename ──
pattern = re.compile(r'^S_(\d+)_D_(\d+)_(\d+)\.wav$')
records = []
parse_errors = []

for f in tqdm(wav_files, desc='Parsing filenames'):
    m = pattern.match(f.name)
    if m:
        sid, did, idx = int(m.group(1)), int(m.group(2)), int(m.group(3))
        records.append({
            'filename': f.name,
            'filepath': str(f),
            'species_id': sid,
            'species': SPECIES_MAP.get(sid, f'Unknown_{sid}'),
            'domain_id': did,
            'domain': f'D{did}',
            'clip_index': idx,
        })
    else:
        parse_errors.append(f.name)

df = pd.DataFrame(records)

print(f'\n✓ Successfully parsed: {len(df):,} / {len(wav_files):,} files')
if parse_errors:
    print(f'✗ Parse errors: {len(parse_errors)} — {parse_errors[:5]}')
print(f'\nSpecies : {df["species_id"].nunique()} unique')
print(f'Domains : {df["domain_id"].nunique()} unique')
df.head(10)

In [ ]:
# ── Summary Statistics ──
print('DataFrame Info:')
print(f'  Shape: {df.shape}')
print(f'  Dtypes:\n{df.dtypes.to_string()}')
print(f'\n  Species IDs present: {sorted(df["species_id"].unique())}')
print(f'  Domain IDs present : {sorted(df["domain_id"].unique())}')
print(f'\nDescriptive Stats:')
df.describe(include='all')

---
## 2. Univariate Analysis
### 2.1 Species Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Bar chart ──
species_counts = df['species'].value_counts().sort_index()
colors = sns.color_palette('husl', n_colors=len(species_counts))
bars = axes[0].barh(species_counts.index, species_counts.values, color=colors)
axes[0].set_xlabel('Number of Clips')
axes[0].set_title('Species Distribution')
for bar, val in zip(bars, species_counts.values):
    axes[0].text(val + 200, bar.get_y() + bar.get_height()/2, 
                 f'{val:,}', va='center', fontsize=9)

# ── Pie chart ──
axes[1].pie(species_counts.values, labels=species_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Species Proportion')

plt.tight_layout()
plt.savefig('species_distribution.png', bbox_inches='tight')
plt.show()

# Imbalance ratio
ir = species_counts.max() / species_counts.min()
print(f'\nImbalance Ratio (max/min): {ir:.1f}x')
print(f'Majority class: {species_counts.idxmax()} ({species_counts.max():,})')
print(f'Minority class: {species_counts.idxmin()} ({species_counts.min():,})')

### 2.2 Domain Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

domain_counts = df['domain'].value_counts().sort_index()
dcolors = sns.color_palette('Set2', n_colors=len(domain_counts))

# ── Bar chart ──
bars = axes[0].bar(domain_counts.index, domain_counts.values, color=dcolors, edgecolor='black')
axes[0].set_ylabel('Number of Clips')
axes[0].set_title('Domain Distribution')
for bar, val in zip(bars, domain_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 500,
                 f'{val:,}', ha='center', fontsize=10, fontweight='bold')

# ── Log scale ──
axes[1].bar(domain_counts.index, domain_counts.values, color=dcolors, edgecolor='black')
axes[1].set_yscale('log')
axes[1].set_ylabel('Number of Clips (log)')
axes[1].set_title('Domain Distribution (Log Scale)')

plt.tight_layout()
plt.savefig('domain_distribution.png', bbox_inches='tight')
plt.show()

d_ir = domain_counts.max() / domain_counts.min()
print(f'\nDomain Imbalance Ratio: {d_ir:.1f}x')
print(f'Largest domain : {domain_counts.idxmax()} ({domain_counts.max():,} clips)')
print(f'Smallest domain: {domain_counts.idxmin()} ({domain_counts.min():,} clips)')
print(f'\nDomain breakdown:\n{domain_counts.to_string()}')

### 2.3 Audio Duration Analysis

In [ ]:
# ── Sample N files for duration analysis (full scan takes too long) ──
N_SAMPLE = min(5000, len(df))
np.random.seed(42)
sample_idx = np.random.choice(len(df), N_SAMPLE, replace=False)
sample_df = df.iloc[sample_idx].copy()

durations = []
sample_rates = []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc='Loading audio durations'):
    try:
        y, sr = librosa.load(row['filepath'], sr=None, duration=2.0)  # peek only
        dur = librosa.get_duration(filename=row['filepath'])
        durations.append(dur)
        sample_rates.append(sr)
    except Exception as e:
        durations.append(np.nan)
        sample_rates.append(np.nan)

sample_df['duration_s'] = durations
sample_df['sample_rate'] = sample_rates

print(f'Duration stats (N={N_SAMPLE} samples):')
print(sample_df['duration_s'].describe())
print(f'\nSample rate distribution:')
print(sample_df['sample_rate'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Duration histogram
sample_df['duration_s'].dropna().hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(sample_df['duration_s'].median(), color='red', ls='--', label=f'Median={sample_df["duration_s"].median():.2f}s')
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Count')
axes[0].set_title('Clip Duration Distribution')
axes[0].legend()

# Duration by domain
sample_df.dropna(subset=['duration_s']).boxplot(column='duration_s', by='domain', ax=axes[1])
axes[1].set_title('Duration by Domain')
axes[1].set_ylabel('Duration (seconds)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('duration_analysis.png', bbox_inches='tight')
plt.show()

---
## 3. Bivariate & Multivariate Analysis
### 3.1 Species × Domain Interaction Heatmap

In [ ]:
# ── Species × Domain crosstab ──
cross = pd.crosstab(df['species'], df['domain'])
cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts
sns.heatmap(cross, annot=True, fmt=',', cmap='YlOrRd', ax=axes[0],
            linewidths=0.5, cbar_kws={'label': 'Clip Count'})
axes[0].set_title('Species × Domain — Raw Counts')
axes[0].set_ylabel('Species')

# Percentage (row-normalized)
sns.heatmap(cross_pct, annot=True, fmt='.1f', cmap='YlGnBu', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': '% of Species'})
axes[1].set_title('Species × Domain — Row Normalized (%)')
axes[1].set_ylabel('Species')

plt.tight_layout()
plt.savefig('species_domain_heatmap.png', bbox_inches='tight')
plt.show()

# ── Identify missing combinations ──
print('\nMissing Species×Domain combinations (0 clips):')
missing = []
for sp in cross.index:
    for dom in cross.columns:
        if cross.loc[sp, dom] == 0:
            missing.append(f'  {sp} × {dom}')
if missing:
    print('\n'.join(missing))
else:
    print('  None — all combinations present')

print(f'\n✓ Full crosstab:\n{cross.to_string()}')

### 3.2 Domain Coverage per Species

In [ ]:
# How many domains does each species appear in?
domain_coverage = df.groupby('species')['domain_id'].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(domain_coverage.index, domain_coverage.values, 
               color=sns.color_palette('viridis', len(domain_coverage)))
ax.set_xlabel('Number of Domains Present')
ax.set_title('Domain Coverage per Species')
ax.set_xlim(0, 6)
for bar, val in zip(bars, domain_coverage.values):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val}/5', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('domain_coverage.png', bbox_inches='tight')
plt.show()

print('\nSpecies with limited domain coverage (potential generalization risk):')
for sp, n in domain_coverage.items():
    if n < 5:
        print(f'  ⚠ {sp}: only {n}/5 domains')

### 3.3 Spectral Feature Extraction & Comparison

In [ ]:
# ── Extract mel-spectrogram features from a stratified sample ──
# Match baseline: sr=8000, n_fft=512, hop=80 (10ms), n_mels=64

TARGET_SR = 8000
N_FFT = 512
HOP_LENGTH = 80  # 10ms at 8kHz
N_MELS = 64
F_MAX = 4000

N_SPECTRAL = min(500, len(df))  # samples per analysis

# Stratified sample: equal from each species×domain
spectral_sample = df.groupby(['species_id', 'domain_id'], group_keys=False).apply(
    lambda x: x.sample(min(len(x), max(1, N_SPECTRAL // (9*5))), random_state=42)
)
spectral_sample = spectral_sample.reset_index(drop=True)
print(f'Spectral sample: {len(spectral_sample)} clips')

def extract_features(filepath):
    """Extract feature vector matching baseline preprocessing."""
    try:
        y, sr = librosa.load(filepath, sr=TARGET_SR)
        # Peak normalization
        if np.max(np.abs(y)) > 0:
            y = y / np.max(np.abs(y))
        
        mel = librosa.feature.melspectrogram(
            y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, fmax=F_MAX
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)
        
        # Summary stats: mean & std per mel band
        feat = np.concatenate([
            log_mel.mean(axis=1),  # 64 mel-band means
            log_mel.std(axis=1),   # 64 mel-band stds
            [librosa.feature.spectral_centroid(y=y, sr=TARGET_SR).mean()],
            [librosa.feature.spectral_bandwidth(y=y, sr=TARGET_SR).mean()],
            [librosa.feature.spectral_rolloff(y=y, sr=TARGET_SR).mean()],
            [librosa.feature.zero_crossing_rate(y).mean()],
            [np.sqrt(np.mean(y**2))],  # RMS
        ])
        return feat
    except:
        return None

features = []
valid_rows = []
for i, row in tqdm(spectral_sample.iterrows(), total=len(spectral_sample), desc='Extracting features'):
    feat = extract_features(row['filepath'])
    if feat is not None:
        features.append(feat)
        valid_rows.append(i)

X = np.array(features)
feat_df = spectral_sample.loc[valid_rows].reset_index(drop=True)
print(f'\nFeature matrix shape: {X.shape}  (samples × features)')

### 3.4 Mel-Spectrogram Gallery — One per Species

In [ ]:
n_species = df['species_id'].nunique()
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (sid, sp_name) in enumerate(sorted(SPECIES_MAP.items())):
    sp_files = df[df['species_id'] == sid]['filepath'].values
    if len(sp_files) == 0:
        continue
    
    fp = sp_files[np.random.randint(len(sp_files))]
    y, sr = librosa.load(fp, sr=TARGET_SR)
    if np.max(np.abs(y)) > 0:
        y = y / np.max(np.abs(y))
    
    mel = librosa.feature.melspectrogram(
        y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmax=F_MAX
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    
    ax = axes[idx]
    librosa.display.specshow(log_mel, sr=TARGET_SR, hop_length=HOP_LENGTH,
                            x_axis='time', y_axis='mel', ax=ax, fmax=F_MAX)
    ax.set_title(f'S{sid}: {sp_name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Log-Mel Spectrograms — One Random Sample per Species\n(8kHz, 64 mel bands, baseline config)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('spectrogram_gallery.png', bbox_inches='tight')
plt.show()

### 3.5 Domain Shift Visualization — t-SNE & PCA

In [ ]:
# ── Standardize features ──
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── PCA ──
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# ── t-SNE ──
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

fig, axes = plt.subplots(2, 2, figsize=(18, 16))

# PCA colored by species
for sid in sorted(feat_df['species_id'].unique()):
    mask = feat_df['species_id'] == sid
    axes[0, 0].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                       label=SPECIES_MAP[sid], alpha=0.6, s=20)
axes[0, 0].set_title('PCA — Colored by Species')
axes[0, 0].legend(fontsize=7, loc='best')
axes[0, 0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[0, 0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')

# PCA colored by domain
for did in sorted(feat_df['domain_id'].unique()):
    mask = feat_df['domain_id'] == did
    axes[0, 1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                       label=f'D{did}', alpha=0.6, s=20, marker='o')
axes[0, 1].set_title('PCA — Colored by Domain (DOMAIN SHIFT CHECK)')
axes[0, 1].legend(fontsize=9)
axes[0, 1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
axes[0, 1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')

# t-SNE colored by species
for sid in sorted(feat_df['species_id'].unique()):
    mask = feat_df['species_id'] == sid
    axes[1, 0].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                       label=SPECIES_MAP[sid], alpha=0.6, s=20)
axes[1, 0].set_title('t-SNE — Colored by Species')
axes[1, 0].legend(fontsize=7, loc='best')

# t-SNE colored by domain
for did in sorted(feat_df['domain_id'].unique()):
    mask = feat_df['domain_id'] == did
    axes[1, 1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                       label=f'D{did}', alpha=0.6, s=20, marker='o')
axes[1, 1].set_title('t-SNE — Colored by Domain (DOMAIN SHIFT CHECK)')
axes[1, 1].legend(fontsize=9)

plt.suptitle('Feature Space Projections — Domain Shift Analysis\n'
             'If domain clusters separate more than species: model learns domain artifacts, not species cues',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('domain_shift_tsne_pca.png', bbox_inches='tight')
plt.show()

print('\n💡 INTERPRETATION:')
print('  • If RIGHT-side plots show tighter clusters than LEFT-side → domain shift is dominant')
print('  • This means the model risks learning DOMAIN identity instead of SPECIES identity')
print('  • This explains BA_seen=88% but BA_unseen=17.5% in the baseline')

### 3.6 Mean Mel-Spectrum per Domain (Same Species)

In [ ]:
# Show how the SAME species sounds DIFFERENT across domains
# Pick the species with most domain coverage
best_coverage_sp = df.groupby('species_id')['domain_id'].nunique().idxmax()
sp_name = SPECIES_MAP[best_coverage_sp]
sp_domains = sorted(df[df['species_id'] == best_coverage_sp]['domain_id'].unique())

fig, axes = plt.subplots(1, len(sp_domains), figsize=(4*len(sp_domains), 4))
if len(sp_domains) == 1:
    axes = [axes]

for i, did in enumerate(sp_domains):
    subset = df[(df['species_id'] == best_coverage_sp) & (df['domain_id'] == did)]
    # Average mel spectrogram from 10 random clips
    sample_fps = subset['filepath'].values[:10]
    mels = []
    for fp in sample_fps:
        y, sr = librosa.load(fp, sr=TARGET_SR)
        if np.max(np.abs(y)) > 0:
            y = y / np.max(np.abs(y))
        mel = librosa.feature.melspectrogram(
            y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, fmax=F_MAX
        )
        mels.append(librosa.power_to_db(mel, ref=np.max))
    
    min_t = min(m.shape[1] for m in mels)
    mels_aligned = np.array([m[:, :min_t] for m in mels])
    avg_mel = mels_aligned.mean(axis=0)
    
    librosa.display.specshow(avg_mel, sr=TARGET_SR, hop_length=HOP_LENGTH,
                            x_axis='time', y_axis='mel', ax=axes[i], fmax=F_MAX)
    axes[i].set_title(f'D{did} (n={len(subset):,})', fontsize=11)

plt.suptitle(f'Average Mel-Spectrogram of {sp_name} Across Domains\n'
             f'Same species, different recording conditions = DOMAIN SHIFT',
             fontsize=12, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('domain_shift_spectrograms.png', bbox_inches='tight')
plt.show()

---
## 4. Cleaning & Preprocessing Insights
### 4.1 Imbalance Quantification

In [ ]:
# ── Comprehensive imbalance report ──
print('=' * 70)
print('IMBALANCE REPORT')
print('=' * 70)

# Species imbalance
sc = df['species'].value_counts()
print(f'\n▸ SPECIES IMBALANCE')
print(f'  Max/Min ratio: {sc.max()/sc.min():.1f}x')
print(f'  Gini coefficient: {1 - (sc / sc.sum()).apply(lambda x: x**2).sum():.3f}')
print(f'  Effective number of classes (ENS): {1 / (sc / sc.sum()).apply(lambda x: x**2).sum():.2f} / 9')

# Domain imbalance
dc = df['domain'].value_counts()
print(f'\n▸ DOMAIN IMBALANCE')
print(f'  Max/Min ratio: {dc.max()/dc.min():.1f}x')
print(f'  D5 contains {dc.get("D5", 0)/len(df)*100:.1f}% of all data')

# Joint imbalance
joint = df.groupby(['species', 'domain']).size()
print(f'\n▸ JOINT (Species × Domain) IMBALANCE')
print(f'  Non-empty cells : {(joint > 0).sum()} / {9 * 5} = {(joint > 0).sum() / 45 * 100:.0f}%')
print(f'  Max cell         : {joint.max():,} clips')
print(f'  Min non-zero cell: {joint[joint > 0].min():,} clips')
print(f'  Joint max/min    : {joint.max() / joint[joint > 0].min():.0f}x')

print(f'\n▸ TRAINING RECOMMENDATION')
print(f'  Domain-balanced sampling is CRITICAL to prevent domain D5 from dominating')
print(f'  Consider: domain-balanced + species-stratified mini-batches')
print(f'  Papers suggest: sample each batch with equal domain representation')

### 4.2 Spectral Statistics by Domain & Species

In [ ]:
# Mean mel-band energy per domain
mel_means = X[:, :N_MELS]  # first 64 features are mel-band means

fig, ax = plt.subplots(figsize=(14, 5))
for did in sorted(feat_df['domain_id'].unique()):
    mask = feat_df['domain_id'] == did
    avg = mel_means[mask].mean(axis=0)
    ax.plot(range(N_MELS), avg, label=f'D{did}', linewidth=2)

ax.set_xlabel('Mel Band Index')
ax.set_ylabel('Mean Log-Mel Energy (dB)')
ax.set_title('Average Mel-Band Energy Profile by Domain\nDivergence = Evidence of Domain Shift')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mel_energy_by_domain.png', bbox_inches='tight')
plt.show()

# Same but by species
fig, ax = plt.subplots(figsize=(14, 5))
for sid in sorted(feat_df['species_id'].unique()):
    mask = feat_df['species_id'] == sid
    avg = mel_means[mask].mean(axis=0)
    ax.plot(range(N_MELS), avg, label=SPECIES_MAP[sid], linewidth=1.5, alpha=0.8)

ax.set_xlabel('Mel Band Index')
ax.set_ylabel('Mean Log-Mel Energy (dB)')
ax.set_title('Average Mel-Band Energy Profile by Species')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mel_energy_by_species.png', bbox_inches='tight')
plt.show()

### 4.3 Statistical Domain Shift Test

In [ ]:
# ── Kruskal-Wallis test: are mel features significantly different across domains? ──
print('Kruskal-Wallis Test: Are features statistically different across domains?')
print('H₀: All domains produce similar feature distributions')
print('H₁: At least one domain differs significantly')
print('-' * 55)

n_significant = 0
domain_groups = {d: mel_means[feat_df['domain_id'] == d] for d in sorted(feat_df['domain_id'].unique())}

for band_i in range(0, N_MELS, 8):  # Test every 8th band
    groups = [domain_groups[d][:, band_i] for d in domain_groups if len(domain_groups[d]) > 0]
    if len(groups) >= 2:
        stat, p = stats.kruskal(*groups)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        if p < 0.05:
            n_significant += 1
        print(f'  Mel band {band_i:2d}: H={stat:8.1f}, p={p:.2e}  {sig}')

print(f'\n→ {n_significant}/{N_MELS//8} tested bands show significant domain differences')
print('  This confirms DOMAIN SHIFT is a real and measurable problem in this dataset.')

### 4.4 Summary & Key Findings

In [ ]:
print('=' * 70)
print('EDA SUMMARY — BioDCASE 2026 Cross-Domain Mosquito Classification')
print('=' * 70)

print(f'''
📊 DATASET OVERVIEW
   Total clips  : {len(df):,}
   Species      : {df['species_id'].nunique()}
   Domains      : {df['domain_id'].nunique()}
   
⚠️  KEY CHALLENGES IDENTIFIED
   1. EXTREME DOMAIN IMBALANCE
      • D5 dominates with ~{dc.get('D5', 0):,} clips ({dc.get('D5', 0)/len(df)*100:.1f}% of data)
      • Domain max/min ratio: {dc.max()/dc.min():.0f}x
      
   2. SPECIES IMBALANCE  
      • Species max/min ratio: {sc.max()/sc.min():.1f}x
      
   3. DOMAIN SHIFT (THE core challenge)
      • PCA/t-SNE show domain clusters are more separated than species clusters
      • Statistical tests confirm mel-band distributions differ significantly across domains
      • Baseline: BA_seen=88.06% vs BA_unseen=17.51% (70-point gap!)

🎯 STRATEGY TO BEAT BASELINE
   1. Domain-Balanced Sampling → equalize domain representation in each mini-batch
   2. Domain-Adversarial Training → learn domain-INVARIANT features
   3. Contrastive Learning (DR-BioL) → push same-species embeddings together regardless of domain
   4. Heavy Augmentation → pitch shift, time stretch, noise addition, SpecAugment
   5. Training-set-only normalization → prevent test-domain leakage
''')

print('✅ EDA Complete. Next step: Domain-Balanced Sampler + Training Pipeline.')